# **Import and parameters**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, IntegerType, LongType, ShortType, ByteType,
    DoubleType, FloatType, DecimalType, BooleanType,
    DateType, TimestampType, BinaryType
)

spark = SparkSession.builder.getOrCreate()

# -------------------------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------------------------
SILVER_LAKEHOUSE = "LH_Silver"      
SAMPLE_FRACTION  = 1.0          # full scan
STRING_PADDING   = 1.20
MIN_VARCHAR_LEN  = 10
USE_UNICODE = False             # Fabric Warehouse: use VARCHAR

# **Functions to detect datatype**

In [ ]:
def string_profile(df, col_name):
    """
    Analyzes the maximum length of a string column and estimates an appropriate VARCHAR(n) or NVARCHAR(n) length,
    adding a configurable safety margin. Returns a warning for very long text columns.
    """
    stats = df.select(F.max(F.length(F.col(col_name))).alias("max_len")).collect()[0]
    max_len = stats["max_len"] or MIN_VARCHAR_LEN
    lunghezza = max(int(max_len * STRING_PADDING), MIN_VARCHAR_LEN)
    prefisso = "NVARCHAR" if USE_UNICODE else "VARCHAR"
    if lunghezza > 4000:
        return f"{prefisso}(MAX)", "⚠ testo lungo, valutare se necessario in Gold"
    return f"{prefisso}({lunghezza})", None

def decimal_profile(df, col_name):
    """
    Analyzes the maximum absolute value in a numeric column to estimate the required decimal precision. 
    Returns a DECIMAL(p,4) type, reserving four decimal places.
    """
    stats = df.select(F.max(F.abs(F.col(col_name))).alias("max_val")).collect()[0]
    max_val = stats["max_val"] or 0
    parte_intera = len(str(int(max_val))) if max_val >= 1 else 1
    scale = 4
    precision = min(parte_intera + scale, 38)
    return f"DECIMAL({precision},{scale})"

def map_datatype(df, col_name, spark_type):
    """
    Maps each Spark data type to a corresponding SQL Warehouse data type (e.g., IntegerType → INT, TimestampType → DATETIME2(6)).
    It delegates string and floating-point columns to the profiling functions and returns warnings when necessary.
    """
    warning = None
    if isinstance(spark_type, StringType):
        sql_type, warning = string_profile(df, col_name)
    elif isinstance(spark_type, (IntegerType, ShortType, ByteType)):
        sql_type = "INT"
    elif isinstance(spark_type, LongType):
        sql_type = "BIGINT"
    elif isinstance(spark_type, (DoubleType, FloatType)):
        sql_type = decimal_profile(df, col_name)
        warning = "convertito da FLOAT/DOUBLE a DECIMAL"
    elif isinstance(spark_type, DecimalType):
        sql_type = f"DECIMAL({spark_type.precision},{spark_type.scale})"
    elif isinstance(spark_type, BooleanType):
        sql_type = "BIT"
    elif isinstance(spark_type, DateType):
        sql_type = "DATE"
    elif isinstance(spark_type, TimestampType):
        sql_type = "DATETIME2(6)"
    elif isinstance(spark_type, BinaryType):
        sql_type = "VARBINARY(MAX)"
    else:
        sql_type = "VARCHAR(255)"
        warning = f"tipo non mappato esplicitamente ({spark_type}), fallback"
    return sql_type, warning

# **Functions data quality**

In [ ]:
def calcola_null_per_colonna(df, col_name, total_rows):
    """
    It returns teh count and percentage of NULL values for the specified column
    """
    null_count = df.filter(F.col(col_name).isNull()).count()
    pct = round((null_count / total_rows) * 100, 2) if total_rows > 0 else 0.0
    return null_count, pct

def calcola_duplicati_tabella(df, total_rows):
    """
    It verifies the presence of duplicated rows considering all columns.
    It returns number of rows duplicated and the percentage of the total.
    """
    distinct_rows = df.distinct().count()
    duplicati = total_rows - distinct_rows
    pct = round((duplicati / total_rows) * 100, 2) if total_rows > 0 else 0.0
    return duplicati, pct

# **Orchestration**

In [ ]:
def genera_report_completo(tabelle=None):
    if tabelle is None:
        tabelle = [row.tableName for row in spark.sql(f"SHOW TABLES IN {SILVER_LAKEHOUSE}").collect()]

    risultati = {}

    for tabella in tabelle:
        print(f"🔍 Analisi tabella: {tabella}")
        df = spark.table(f"{SILVER_LAKEHOUSE}.{tabella}")
        df_analisi = df if SAMPLE_FRACTION >= 1.0 else df.sample(fraction=SAMPLE_FRACTION, seed=42)

        # Cache   to avoid multiple recalculation on count/distinct/null
        df_analisi.cache()
        total_rows = df_analisi.count()

        # Duplicated calculated one time a full table level
        duplicati_count, duplicati_pct = calcola_duplicati_tabella(df_analisi, total_rows)
        flag_duplicati = f"SI ({duplicati_count} righe, {duplicati_pct}%)" if duplicati_count > 0 else "NO"

        righe = []
        for field in df_analisi.schema.fields:
            sql_type, warning = map_datatype(df_analisi, field.name, field.dataType)
            null_count, null_pct = calcola_null_per_colonna(df_analisi, field.name, total_rows)
            flag_null = f"SI ({null_count} righe, {null_pct}%)" if null_count > 0 else "NO"

            righe.append({
                "colonna": field.name,
                "tipo_spark": str(field.dataType),
                "tipo_sql_proposto": sql_type,
                "nullable_schema": field.nullable,
                "presenza_null": flag_null,
                "presenza_duplicati_tabella": flag_duplicati,   # Same value repeated per row
                "note": warning or ""
            })

        df_analisi.unpersist()
        risultati[tabella] = righe

    return risultati

# **Execution and print report**

In [ ]:
report = genera_report_completo()

for tabella, colonne in report.items():
    print(f"\n📋 Tabella: {tabella}  |  Duplicati riga intera: {colonne[0]['presenza_duplicati_tabella']}")
    print(f"{'Colonna':<22} {'Tipo Spark':<18} {'Tipo SQL':<20} {'Null Schema':<12} {'Presenza NULL':<22} Note")
    print("-" * 120)
    for c in colonne:
        print(f"{c['colonna']:<22} {c['tipo_spark']:<18} {c['tipo_sql_proposto']:<20} "
              f"{str(c['nullable_schema']):<12} {c['presenza_null']:<22} {c['note']}")

# **Export in CSV**

In [ ]:

import pandas as pd

def report_to_flat_list(report):
    """
    Flatten the report dictionary into a list of records,
    adding the table name as the first column.
    """
    righe_finali = []
    for tabella, colonne in report.items():
        for c in colonne:
            righe_finali.append({
                "tabella": tabella,
                "colonna": c["colonna"],
                "tipo_spark": c["tipo_spark"],
                "tipo_sql_proposto": c["tipo_sql_proposto"],
                "nullable_schema": c["nullable_schema"],
                "presenza_null": c["presenza_null"],
                "presenza_duplicati_tabella": c["presenza_duplicati_tabella"],
                "note": c["note"]
            })
    return righe_finali

# Generate report using previous functions
report = genera_report_completo()
righe_finali = report_to_flat_list(report)

# Convert into Pandas dataframe
pdf = pd.DataFrame(righe_finali)

# -------------------------------------------------------------------------------------
# OUTPUT 1: Print TSV format (Tab-Separated) -> copy/paste directly in Excel
# -------------------------------------------------------------------------------------
print("📋 COPIA IL BLOCCO SOTTOSTANTE E INCOLLALO DIRETTAMENTE IN EXCEL (Ctrl+V):\n")
print(pdf.to_csv(sep="\t", index=False))

# -------------------------------------------------------------------------------------
# OUTPUT 2: Save as CSV file in Onelake Files
# -------------------------------------------------------------------------------------
# 1. Salvataggio temporaneo su filesystem locale del cluster Spark (driver node)
local_temp_path = "/tmp/report_datatype_silver_to_gold.csv"
pdf.to_csv(local_temp_path, sep=";", index=False, encoding="utf-8-sig")

# 2. Path OneLake destination (ABFSS) in the LH_Silver
onelake_folder = "abfss://GlobalRetail@onelake.dfs.fabric.microsoft.com/LH_Silver.Lakehouse/Files/report_datatype_gold" # Indirizzo abfss copiato da Files
onelake_file_path = f"{onelake_folder}/report_datatype_silver_to_gold.csv"

# 3. Create destination folder if it doesn't exist
notebookutils.fs.mkdirs(onelake_folder)

# 4. Copy the file from the local filesystem to Inelake
notebookutils.fs.cp(f"file:{local_temp_path}", onelake_file_path)

print(f"✅ Report salvato correttamente su OneLake in: {onelake_file_path}")
print("   Puoi scaricarlo dall'area 'Files' del Lakehouse Silver (Esplora Lakehouse) e aprirlo in Excel.")
